In [191]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import kagglehub
import os

## Classify

In [192]:
dataset_dir = kagglehub.dataset_download("mssmartypants/water-quality")
print("Path to dataset files:", dataset_dir)

Path to dataset files: /Users/waroonragwongsiri/.cache/kagglehub/datasets/mssmartypants/water-quality/versions/3


In [193]:
for filename in os.listdir(dataset_dir):
	if filename.endswith(".csv"):
		filepath = os.path.join(dataset_dir, filename)
		break

In [194]:
df = pd.read_csv(filepath)
df

,aluminium,ammonia,arsenic,barium,cadmium,chloramine,chromium,copper,flouride,bacteria,...,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe
0,1.65,9.08,0.04,2.85,0.007,0.35,0.83,0.17,0.05,0.20,...,0.054,16.08,1.13,0.007,37.75,6.78,0.08,0.34,0.02,1
1,2.32,21.16,0.01,3.31,0.002,5.28,0.68,0.66,0.90,0.65,...,0.100,2.01,1.93,0.003,32.26,3.21,0.08,0.27,0.05,1
2,1.01,14.02,0.04,0.58,0.008,4.24,0.53,0.02,0.99,0.05,...,0.078,14.16,1.11,0.006,50.28,7.07,0.07,0.44,0.01,0
3,1.36,11.33,0.04,2.96,0.001,7.23,0.03,1.66,1.08,0.71,...,0.016,1.41,1.29,0.004,9.12,1.72,0.02,0.45,0.05,1
4,0.92,24.33,0.03,0.20,0.006,2.67,0.69,0.57,0.61,0.13,...,0.117,6.74,1.11,0.003,16.90,2.41,0.02,0.06,0.02,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7994,0.05,7.78,0.00,1.95,0.040,0.10,0.03,0.03,1.37,0.00,...,0.197,14.29,1.00,0.005,3.57,2.13,0.09,0.06,0.03,1
7995,0.05,24.22,0.02,0.59,0.010,0.45,0.02,0.02,1.48,0.00,...,0.031,10.27,1.00,0.001,1.48,1.11,0.09,0.10,0.08,1
7996,0.09,6.85,0.00,0.61,0.030,0.05,0.05,0.02,0.91,0.00,...,0.182,15.92,1.00,0.000,1.35,4.84,0.00,0.04,0.05,1
7997,0.01,10,0.01,2.00,0.000,2.00,0.00,0.09,0.00,0.00,...,0.000,0.00,0.00,0.000,0.00,0.00,0.00,0.00,0.00,1


In [195]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7999 entries, 0 to 7998
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   aluminium    7999 non-null   float64
 1   ammonia      7999 non-null   str    
 2   arsenic      7999 non-null   float64
 3   barium       7999 non-null   float64
 4   cadmium      7999 non-null   float64
 5   chloramine   7999 non-null   float64
 6   chromium     7999 non-null   float64
 7   copper       7999 non-null   float64
 8   flouride     7999 non-null   float64
 9   bacteria     7999 non-null   float64
 10  viruses      7999 non-null   float64
 11  lead         7999 non-null   float64
 12  nitrates     7999 non-null   float64
 13  nitrites     7999 non-null   float64
 14  mercury      7999 non-null   float64
 15  perchlorate  7999 non-null   float64
 16  radium       7999 non-null   float64
 17  selenium     7999 non-null   float64
 18  silver       7999 non-null   float64
 19  uranium      7999

In [196]:
df['is_safe'].value_counts()

is_safe
0        7084
1         912
#NUM!       3
Name: count, dtype: int64

In [197]:
df = df[df.is_safe != '#NUM!']
df['is_safe'].value_counts()

is_safe
0    7084
1     912
Name: count, dtype: int64

In [198]:
df["is_safe"] = pd.to_numeric(df["is_safe"])
df.info()

<class 'pandas.DataFrame'>
Index: 7996 entries, 0 to 7998
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   aluminium    7996 non-null   float64
 1   ammonia      7996 non-null   str    
 2   arsenic      7996 non-null   float64
 3   barium       7996 non-null   float64
 4   cadmium      7996 non-null   float64
 5   chloramine   7996 non-null   float64
 6   chromium     7996 non-null   float64
 7   copper       7996 non-null   float64
 8   flouride     7996 non-null   float64
 9   bacteria     7996 non-null   float64
 10  viruses      7996 non-null   float64
 11  lead         7996 non-null   float64
 12  nitrates     7996 non-null   float64
 13  nitrites     7996 non-null   float64
 14  mercury      7996 non-null   float64
 15  perchlorate  7996 non-null   float64
 16  radium       7996 non-null   float64
 17  selenium     7996 non-null   float64
 18  silver       7996 non-null   float64
 19  uranium      7996 non-

In [199]:
df["ammonia"] = pd.to_numeric(df["ammonia"])
df.info()

<class 'pandas.DataFrame'>
Index: 7996 entries, 0 to 7998
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   aluminium    7996 non-null   float64
 1   ammonia      7996 non-null   float64
 2   arsenic      7996 non-null   float64
 3   barium       7996 non-null   float64
 4   cadmium      7996 non-null   float64
 5   chloramine   7996 non-null   float64
 6   chromium     7996 non-null   float64
 7   copper       7996 non-null   float64
 8   flouride     7996 non-null   float64
 9   bacteria     7996 non-null   float64
 10  viruses      7996 non-null   float64
 11  lead         7996 non-null   float64
 12  nitrates     7996 non-null   float64
 13  nitrites     7996 non-null   float64
 14  mercury      7996 non-null   float64
 15  perchlorate  7996 non-null   float64
 16  radium       7996 non-null   float64
 17  selenium     7996 non-null   float64
 18  silver       7996 non-null   float64
 19  uranium      7996 non-

In [200]:
columns = [
	"size", "price", "rooms",
	"aluminium", "ammonia", "arsenic", "barium",
	"cadmium", "chlorine", "copper", "fluoride",
	"bacteria", "viruses", "lead", "nitrates",
	"nitrites", "mercury", "perchlorate", "radium",
	"selenium", "silver", "uranium", "is_safe",
	"new_price"
]

# Training set
training_data = [
	[96.0, 0.90, 2, 1.0, 9.1, 0.0, 0.1, 0.0, 1.0, 0.0, 1.2, 0.0, 0.0, 0.1, 0.0, 2.0, 1.0, 0.0, 22.3, 2.2, 0.05, 0.2, 0, 1.1],
	[118.0, 1.25, 2, 0.2, 7.7, 0.0, 0.5, 0.0, 1.2, 0.5, 0.9, 0.0, 0.0, 0.0, 0.0, 1.5, 0.1, 0.0, 19.1, 1.1, 0.1, 0.02, 1, 2.0],
	[141.2, 1.50, 3, 0.1, 5.1, 0.0, 0.1, 0.0, 0.5, 0.2, 0.5, 0.1, 0.1, 0.0, 0.5, 5.5, 0.2, 0.0, 11.5, 0.5, 0.1, 0.05, 0, 2.9],
	[129.6, 1.45, 3, 0.2, 5.3, 0.0, 0.2, 0.0, 1.5, 0.4, 0.6, 0.0, 0.0, 0.0, 3.4, 0.3, 0.0, 0.0, 8.8, 1.2, 0.09, 0.03, 1, 2.9],
	[110.0, 1.15, 2, 0.0, 1.2, 0.0, 0.1, 0.0, 0.9, 0.1, 0.4, 0.0, 0.0, 0.0, 1.0, 0.1, 0.0, 0.0, 7.1, 0.8, 0.1, 0.03, 1, 1.7],
	[105.0, 1.00, 2, 3.0, 10.1, 0.1, 0.6, 0.0, 2.0, 0.0, 1.1, 0.0, 0.0, 0.0, 2.0, 2.1, 1.0, 0.0, 12.9, 3.0, 0.15, 0.04, 0, 1.5],
]

# Testing set
testing_data = [
	[125.0, 1.30, 3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.0, 0.0, 0.0, 0.01, None, None]
]

train_reg_df = pd.DataFrame(data=training_data, columns=columns)
train_reg_df

,size,price,rooms,aluminium,ammonia,arsenic,barium,cadmium,chlorine,copper,...,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe,new_price
0,96.0,0.90,2,1.0,9.1,0.0,0.1,0.0,1.0,0.0,...,0.0,2.0,1.0,0.0,22.3,2.2,0.05,0.20,0,1.1
1,118.0,1.25,2,0.2,7.7,0.0,0.5,0.0,1.2,0.5,...,0.0,1.5,0.1,0.0,19.1,1.1,0.10,0.02,1,2.0
2,141.2,1.50,3,0.1,5.1,0.0,0.1,0.0,0.5,0.2,...,0.5,5.5,0.2,0.0,11.5,0.5,0.10,0.05,0,2.9
3,129.6,1.45,3,0.2,5.3,0.0,0.2,0.0,1.5,0.4,...,3.4,0.3,0.0,0.0,8.8,1.2,0.09,0.03,1,2.9
4,110.0,1.15,2,0.0,1.2,0.0,0.1,0.0,0.9,0.1,...,1.0,0.1,0.0,0.0,7.1,0.8,0.10,0.03,1,1.7
5,105.0,1.00,2,3.0,10.1,0.1,0.6,0.0,2.0,0.0,...,2.0,2.1,1.0,0.0,12.9,3.0,0.15,0.04,0,1.5


In [201]:
possible_x = []

for col in list(X.columns):
	if col in columns:
		possible_x.append(col)

print(possible_x)

['aluminium', 'ammonia', 'arsenic', 'barium', 'cadmium', 'copper', 'bacteria', 'viruses', 'lead', 'nitrates', 'nitrites', 'mercury', 'perchlorate', 'radium', 'selenium', 'silver', 'uranium']


In [202]:
X = df.drop(columns='is_safe')[possible_x]
y = df['is_safe']

In [203]:
X

,aluminium,ammonia,arsenic,barium,cadmium,copper,bacteria,viruses,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium
0,1.65,9.08,0.04,2.85,0.007,0.17,0.20,0.000,0.054,16.08,1.13,0.007,37.75,6.78,0.08,0.34,0.02
1,2.32,21.16,0.01,3.31,0.002,0.66,0.65,0.650,0.100,2.01,1.93,0.003,32.26,3.21,0.08,0.27,0.05
2,1.01,14.02,0.04,0.58,0.008,0.02,0.05,0.003,0.078,14.16,1.11,0.006,50.28,7.07,0.07,0.44,0.01
3,1.36,11.33,0.04,2.96,0.001,1.66,0.71,0.710,0.016,1.41,1.29,0.004,9.12,1.72,0.02,0.45,0.05
4,0.92,24.33,0.03,0.20,0.006,0.57,0.13,0.001,0.117,6.74,1.11,0.003,16.90,2.41,0.02,0.06,0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7994,0.05,7.78,0.00,1.95,0.040,0.03,0.00,0.000,0.197,14.29,1.00,0.005,3.57,2.13,0.09,0.06,0.03
7995,0.05,24.22,0.02,0.59,0.010,0.02,0.00,0.000,0.031,10.27,1.00,0.001,1.48,1.11,0.09,0.10,0.08
7996,0.09,6.85,0.00,0.61,0.030,0.02,0.00,0.000,0.182,15.92,1.00,0.000,1.35,4.84,0.00,0.04,0.05
7997,0.01,10.00,0.01,2.00,0.000,0.09,0.00,0.000,0.000,0.00,0.00,0.000,0.00,0.00,0.00,0.00,0.00


In [204]:
y

0       1
1       1
2       0
3       1
4       1
       ..
7994    1
7995    1
7996    1
7997    1
7998    1
Name: is_safe, Length: 7996, dtype: int64

In [205]:
X.describe()

,aluminium,ammonia,arsenic,barium,cadmium,copper,bacteria,viruses,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium
count,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000,7996.000000
mean,0.666396,14.278212,0.161477,1.567928,0.042803,0.805940,0.319714,0.328706,0.099431,9.819250,1.329846,0.005193,16.465266,2.920106,0.049684,0.147811,0.044672
std,1.265323,8.878930,0.252632,1.216227,0.036049,0.653595,0.329497,0.378113,0.058169,5.541977,0.573271,0.002967,17.688827,2.322805,0.028773,0.143569,0.026906
min,0.000000,-0.080000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.040000,6.577500,0.030000,0.560000,0.008000,0.090000,0.000000,0.002000,0.048000,5.000000,1.000000,0.003000,2.170000,0.820000,0.020000,0.040000,0.020000
50%,0.070000,14.130000,0.050000,1.190000,0.040000,0.750000,0.220000,0.008000,0.102000,9.930000,1.420000,0.005000,7.745000,2.410000,0.050000,0.080000,0.050000
75%,0.280000,22.132500,0.100000,2.482500,0.070000,1.390000,0.610000,0.700000,0.151000,14.610000,1.760000,0.008000,29.487500,4.670000,0.070000,0.240000,0.070000
max,5.050000,29.840000,1.050000,4.940000,0.130000,2.000000,1.000000,1.000000,0.200000,19.830000,2.930000,0.010000,60.010000,7.990000,0.100000,0.500000,0.090000


In [206]:
y.value_counts()

is_safe
0    7084
1     912
Name: count, dtype: int64

In [207]:
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=42, stratify=y)

In [208]:
cls_model = LGBMClassifier(random_state=42, verbose=-1) # no scaling, missing values OK
cls_model.fit(X_train, y_train)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [209]:
y_pred_train = cls_model.predict(X_train)
y_pred_test = cls_model.predict(X_test)

print(f"Train  Acc: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Test   Acc: {accuracy_score(y_test, y_pred_test):.3f}")
print(classification_report(y_test, y_pred_test))

Train  Acc: 1.000
Test   Acc: 0.970
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1418
           1       0.92      0.81      0.86       182

    accuracy                           0.97      1600
   macro avg       0.95      0.90      0.92      1600
weighted avg       0.97      0.97      0.97      1600



In [210]:
X_test.shape

(1600, 17)

In [211]:
y_test.shape

(1600,)

In [212]:
y_pred_test.shape

(1600,)

## Prepare Regression

In [213]:
testing_reg_df = pd.DataFrame(data=testing_data, columns=columns)
testing_reg_df.columns

Index(['size', 'price', 'rooms', 'aluminium', 'ammonia', 'arsenic', 'barium',
       'cadmium', 'chlorine', 'copper', 'fluoride', 'bacteria', 'viruses',
       'lead', 'nitrates', 'nitrites', 'mercury', 'perchlorate', 'radium',
       'selenium', 'silver', 'uranium', 'is_safe', 'new_price'],
      dtype='str')

In [214]:
list(X.columns)

['aluminium',
 'ammonia',
 'arsenic',
 'barium',
 'cadmium',
 'copper',
 'bacteria',
 'viruses',
 'lead',
 'nitrates',
 'nitrites',
 'mercury',
 'perchlorate',
 'radium',
 'selenium',
 'silver',
 'uranium']

In [215]:
tmp_reg_x = testing_reg_df[possible_x]
tmp_reg_x

,aluminium,ammonia,arsenic,barium,cadmium,copper,bacteria,viruses,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01,0.0,0.0,0.0,0.01


In [216]:
tmp_reg_x.shape

(1, 17)

In [217]:
tmp_reg_y = cls_model.predict(tmp_reg_x)
tmp_reg_y

array([1])

In [218]:
train_reg_df

,size,price,rooms,aluminium,ammonia,arsenic,barium,cadmium,chlorine,copper,...,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe,new_price
0,96.0,0.90,2,1.0,9.1,0.0,0.1,0.0,1.0,0.0,...,0.0,2.0,1.0,0.0,22.3,2.2,0.05,0.20,0,1.1
1,118.0,1.25,2,0.2,7.7,0.0,0.5,0.0,1.2,0.5,...,0.0,1.5,0.1,0.0,19.1,1.1,0.10,0.02,1,2.0
2,141.2,1.50,3,0.1,5.1,0.0,0.1,0.0,0.5,0.2,...,0.5,5.5,0.2,0.0,11.5,0.5,0.10,0.05,0,2.9
3,129.6,1.45,3,0.2,5.3,0.0,0.2,0.0,1.5,0.4,...,3.4,0.3,0.0,0.0,8.8,1.2,0.09,0.03,1,2.9
4,110.0,1.15,2,0.0,1.2,0.0,0.1,0.0,0.9,0.1,...,1.0,0.1,0.0,0.0,7.1,0.8,0.10,0.03,1,1.7
5,105.0,1.00,2,3.0,10.1,0.1,0.6,0.0,2.0,0.0,...,2.0,2.1,1.0,0.0,12.9,3.0,0.15,0.04,0,1.5


In [219]:
test_reg_df = pd.DataFrame(data=testing_data, columns=columns)
test_reg_df['is_safe'] = tmp_reg_y
test_reg_df

,size,price,rooms,aluminium,ammonia,arsenic,barium,cadmium,chlorine,copper,...,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe,new_price
0,125.0,1.3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.01,0.0,0.0,0.0,0.01,1,None


## Regression

In [220]:
reg_X_train = train_reg_df.drop(columns='new_price')
reg_X_train

reg_y_train = train_reg_df['new_price']
reg_y_train

reg_X_test = test_reg_df.drop(columns='new_price')
reg_X_test

reg_y_test = test_reg_df['new_price']
reg_y_test

0    None
Name: new_price, dtype: object

In [221]:
reg_model = LGBMRegressor(random_state=42, verbose=-1) # no scaling, missing values OK
reg_model.fit(reg_X_train, reg_y_train)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [222]:
reg_y_pred_train = reg_model.predict(reg_X_train)

print(f"Train  RMSE: {root_mean_squared_error(reg_y_train, reg_y_pred_train):.2f}  R2: {r2_score(reg_y_train, reg_y_pred_train):.3f}")

Train  RMSE: 0.68  R2: -0.000


In [223]:
reg_y_pred_test = reg_model.predict(reg_X_test)
reg_y_pred_test

array([2.01666671])

In [224]:
submission_reg = test_reg_df
submission_reg['new_price'] = reg_y_pred_test
submission_reg

,size,price,rooms,aluminium,ammonia,arsenic,barium,cadmium,chlorine,copper,...,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe,new_price
0,125.0,1.3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.01,0.0,0.0,0.0,0.01,1,2.016667


## Model Score

In [225]:
y_pred_train = cls_model.predict(X_train)
y_pred_test = cls_model.predict(X_test)

print(f"Train  Acc: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Test   Acc: {accuracy_score(y_test, y_pred_test):.3f}")
print(classification_report(y_test, y_pred_test))

Train  Acc: 1.000
Test   Acc: 0.970
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1418
           1       0.92      0.81      0.86       182

    accuracy                           0.97      1600
   macro avg       0.95      0.90      0.92      1600
weighted avg       0.97      0.97      0.97      1600



In [226]:
reg_y_pred_train = reg_model.predict(reg_X_train)

print(f"Train  RMSE: {root_mean_squared_error(reg_y_train, reg_y_pred_train):.2f}  R2: {r2_score(reg_y_train, reg_y_pred_train):.3f}")

Train  RMSE: 0.68  R2: -0.000
